In [2]:
import pandas as pd
import numpy as np

def load_data(filepath):
    print("=" * 50)
    print("LOADING DATA")
    print("=" * 50)
    
    # Check if file exists
    import os
    if not os.path.exists(filepath):
        print("ERROR: File not found. Please check your filepath.")
        return None
    
    # Check file extension
    allowed_extensions = ['.csv', '.xlsx', '.xls']
    file_extension = os.path.splitext(filepath)[1].lower()
    
    if file_extension not in allowed_extensions:
        print(f"ERROR: Unsupported file type '{file_extension}'.")
        print(f"Supported formats: CSV, Excel (.xlsx, .xls)")
        return None
    
    # Try loading the file
    try:
        if file_extension == '.csv':
            df = pd.read_csv(filepath)
            print(f"SUCCESS: CSV file loaded.")
        elif file_extension in ['.xlsx', '.xls']:
            df = pd.read_excel(filepath)
            print(f"SUCCESS: Excel file loaded.")
        
        # Check if file is empty
        if df.empty:
            print("ERROR: File is empty. Please upload a file with data.")
            return None
        
        # Check if file has at least 2 columns
        if df.shape[1] < 2:
            print("ERROR: File has less than 2 columns. Not enough data to analyse.")
            return None
        
        print(f"Rows loaded: {df.shape[0]}")
        print(f"Columns loaded: {df.shape[1]}")
        print("=" * 50)
        return df
    
    except Exception as e:
        print(f"ERROR: Could not read file. Reason: {str(e)}")
        return None

# Test it with your file
df = load_data('C:/SupplyChainAssistant/data/supply_chain.csv')

if df is not None:
    df_clean = df.copy()
    print("Working copy created successfully")

LOADING DATA
ERROR: File not found. Please check your filepath.


In [3]:
df = load_data('C:/SupplyChainAssistant/data/supply_chain_data.csv')

if df is not None:
    df_clean = df.copy()
    print("Working copy created successfully")

LOADING DATA
SUCCESS: CSV file loaded.
Rows loaded: 100
Columns loaded: 24
Working copy created successfully


In [4]:
def profile_data(df):
    print("=" * 50)
    print("DATA PROFILE REPORT")
    print("=" * 50)
    
    # Basic shape
    print(f"\nTotal Rows: {df.shape[0]}")
    print(f"Total Columns: {df.shape[1]}")
    
    # Duplicates
    print(f"Duplicate Rows: {df.duplicated().sum()}")
    
    # Missing values
    missing = df.isnull().sum()
    missing_cols = missing[missing > 0]
    if len(missing_cols) == 0:
        print("Missing Values: None found")
    else:
        print(f"\nMissing Values Found:")
        for col, count in missing_cols.items():
            pct = (count / len(df)) * 100
            print(f"  {col}: {count} missing ({pct:.1f}%)")
    
    # Dtypes summary
    print(f"\nNumeric Columns: {df.select_dtypes(include='number').shape[1]}")
    print(f"Categorical Columns: {df.select_dtypes(include='object').shape[1]}")
    
    print("\n" + "=" * 50)

# Test it
profile_data(df_clean)

DATA PROFILE REPORT

Total Rows: 100
Total Columns: 24
Duplicate Rows: 0
Missing Values: None found

Numeric Columns: 15
Categorical Columns: 9



In [8]:
def profile_data(df):
    print("=" * 50)
    print("DATA PROFILE REPORT")
    print("=" * 50)
    
    # Basic shape
    print(f"\nTotal Rows: {df.shape[0]}")
    print(f"Total Columns: {df.shape[1]}")
    
    # Duplicates
    print(f"Duplicate Rows: {df.duplicated().sum()}")
    
    # Missing values
    missing = df.isnull().sum()
    missing_cols = missing[missing > 0]
    if len(missing_cols) == 0:
        print("Missing Values: None found")
    else:
        print(f"\nMissing Values Found:")
        for col, count in missing_cols.items():
            pct = (count / len(df)) * 100
            print(f"  {col}: {count} missing ({pct:.1f}%)")
    
    # Dtypes summary
    print(f"\nNumeric Columns: {df.select_dtypes(include='number').shape[1]}")
    print(f"Categorical Columns: {df.select_dtypes(include='object').shape[1]}")
    
    # ID column detection
    print("\n--- ID Column Check ---")
    for col in df.columns:
        if df[col].nunique() == len(df) and df[col].dtype == 'object':
            print(f"  WARNING: '{col}' has {df[col].nunique()} unique values in {len(df)} rows - likely an ID column, not useful for analysis")
        
    if not any(df[col].nunique() == len(df) and df[col].dtype == 'object' 
               for col in df.columns):
        print("  No ID columns detected")
    
    # Suspicious values in categorical columns
    print("\n--- Suspicious Value Check ---")
    suspicious = ['unknown', 'n/a', 'na', 'none', 'null', 
                  'not available', 'missing', 'undefined']
    for col in df.select_dtypes(include='object').columns:
        for val in df[col].unique():
            if str(val).lower().strip() in suspicious:
                count = df[col].value_counts()[val]
                pct = (count / len(df)) * 100
                print(f"  WARNING: '{col}' contains '{val}' in {count} rows ({pct:.1f}%) - treat as missing value")
    
    # Zero value check in numeric columns
    print("\n--- Zero Value Check ---")
    for col in df.select_dtypes(include='number').columns:
        zero_count = (df[col] == 0).sum()
        if zero_count > 0:
            print(f"  WARNING: '{col}' has {zero_count} zero values - verify if valid or missing")
    
    # Categorical column details
    print("\n--- Categorical Column Details ---")
    for col in df.select_dtypes(include='object').columns:
        if df[col].nunique() == len(df):
            print(f"\n  {col}: Skipped - flagged as ID column")
            continue
        print(f"\n  {col} ({df[col].nunique()} unique values):")
        for val, count in df[col].value_counts().items():
            pct = (count / len(df)) * 100
            print(f"    {val}: {count} ({pct:.1f}%)")
    
    print("\n" + "=" * 50)
    print("END OF PROFILE REPORT")
    print("=" * 50)

# Test it
profile_data(df_clean)

DATA PROFILE REPORT

Total Rows: 100
Total Columns: 24
Duplicate Rows: 0
Missing Values: None found

Numeric Columns: 15
Categorical Columns: 9

--- ID Column Check ---

--- Suspicious Value Check ---

--- Zero Value Check ---

--- Categorical Column Details ---

  Product type (3 unique values):
    skincare: 40 (40.0%)
    haircare: 34 (34.0%)
    cosmetics: 26 (26.0%)

  SKU: Skipped - flagged as ID column

  Customer demographics (4 unique values):
    Unknown: 31 (31.0%)
    Female: 25 (25.0%)
    Non-binary: 23 (23.0%)
    Male: 21 (21.0%)

  Shipping carriers (3 unique values):
    Carrier B: 43 (43.0%)
    Carrier C: 29 (29.0%)
    Carrier A: 28 (28.0%)

  Supplier name (5 unique values):
    Supplier 1: 27 (27.0%)
    Supplier 2: 22 (22.0%)
    Supplier 5: 18 (18.0%)
    Supplier 4: 18 (18.0%)
    Supplier 3: 15 (15.0%)

  Location (5 unique values):
    Kolkata: 25 (25.0%)
    Mumbai: 22 (22.0%)
    Chennai: 20 (20.0%)
    Bangalore: 18 (18.0%)
    Delhi: 15 (15.0%)

  Inspec

In [9]:
def suggest_fixes(df):
    print("=" * 50)
    print("SUGGESTED FIXES")
    print("=" * 50)
    
    fixes_applied = []
    df_fixed = df.copy()
    
    # Suggestion 1 - Suspicious values like Unknown
    print("\n--- Suspicious Values ---")
    suspicious = ['unknown', 'n/a', 'na', 'none', 'null', 
                  'not available', 'missing', 'undefined']
    
    for col in df_fixed.select_dtypes(include='object').columns:
        for val in df_fixed[col].unique():
            if str(val).lower().strip() in suspicious:
                count = (df_fixed[col] == val).sum()
                pct = (count / len(df_fixed)) * 100
                print(f"\n  Found '{val}' in '{col}' column")
                print(f"  Count: {count} rows ({pct:.1f}%)")
                most_frequent = df_fixed[df_fixed[col] != val][col].mode()[0]
                print(f"  Suggestion: Replace with most frequent value '{most_frequent}'")
                choice = input("  Apply fix? (yes/no): ").strip().lower()
                if choice == 'yes':
                    df_fixed[col] = df_fixed[col].replace(val, most_frequent)
                    fixes_applied.append(f"Replaced '{val}' in '{col}' with '{most_frequent}'")
                    print(f"  ✓ Fixed!")
                else:
                    print(f"  Skipped.")
    
    # Suggestion 2 - ID columns
    print("\n--- ID Columns ---")
    id_cols_found = False
    for col in df_fixed.select_dtypes(include='object').columns:
        if df_fixed[col].nunique() == len(df_fixed):
            id_cols_found = True
            print(f"\n  '{col}' appears to be an ID column ({df_fixed[col].nunique()} unique values)")
            print(f"  Suggestion: Drop this column - not useful for analysis")
            choice = input("  Drop column? (yes/no): ").strip().lower()
            if choice == 'yes':
                df_fixed = df_fixed.drop(columns=[col])
                fixes_applied.append(f"Dropped ID column '{col}'")
                print(f"  ✓ Dropped!")
            else:
                print(f"  Skipped.")
    if not id_cols_found:
        print("  No ID columns found.")
    
    # Suggestion 3 - Zero values in numeric columns
    print("\n--- Zero Values ---")
    zero_found = False
    for col in df_fixed.select_dtypes(include='number').columns:
        zero_count = (df_fixed[col] == 0).sum()
        if zero_count > 0:
            zero_found = True
            print(f"\n  '{col}' has {zero_count} zero value(s)")
            print(f"  Suggestion: Replace zeros with median value")
            median_val = df_fixed[df_fixed[col] != 0][col].median()
            print(f"  Median value: {median_val:.2f}")
            choice = input("  Apply fix? (yes/no): ").strip().lower()
            if choice == 'yes':
                df_fixed[col] = df_fixed[col].replace(0, median_val)
                fixes_applied.append(f"Replaced 0 in '{col}' with median {median_val:.2f}")
                print(f"  ✓ Fixed!")
            else:
                print(f"  Skipped.")
    if not zero_found:
        print("  No zero values found.")
    
    # Summary of what was done
    print("\n" + "=" * 50)
    print("FIXES SUMMARY")
    print("=" * 50)
    if fixes_applied:
        print(f"\nTotal fixes applied: {len(fixes_applied)}")
        for fix in fixes_applied:
            print(f"  ✓ {fix}")
    else:
        print("\nNo fixes applied.")
    
    print(f"\nDataset shape before: {df.shape}")
    print(f"Dataset shape after:  {df_fixed.shape}")
    print("=" * 50)
    
    return df_fixed

# Run it
df_clean = suggest_fixes(df_clean)

SUGGESTED FIXES

--- Suspicious Values ---

  Found 'Unknown' in 'Customer demographics' column
  Count: 31 rows (31.0%)
  Suggestion: Replace with most frequent value 'Female'


  Apply fix? (yes/no):  yes


  ✓ Fixed!

--- ID Columns ---

  'SKU' appears to be an ID column (100 unique values)
  Suggestion: Drop this column - not useful for analysis


  Drop column? (yes/no):  yes


  ✓ Dropped!

--- Zero Values ---

  'Stock levels' has 1 zero value(s)
  Suggestion: Replace zeros with median value
  Median value: 48.00


  Apply fix? (yes/no):  yes


  ✓ Fixed!

FIXES SUMMARY

Total fixes applied: 3
  ✓ Replaced 'Unknown' in 'Customer demographics' with 'Female'
  ✓ Dropped ID column 'SKU'
  ✓ Replaced 0 in 'Stock levels' with median 48.00

Dataset shape before: (100, 24)
Dataset shape after:  (100, 23)
